---
title: "最小生成树——Kruskal & Prim 算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [7]:
#| code-fold: true
from typing import List
import heapq


class UnionFind:
    def __init__(self, n):
        self.count = n
        self.parent = list(range(n))

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        find_x = self.find(x)
        find_y = self.find(y)
        if find_x != find_y:
            self.parent[find_x] = find_y
            self.count -= 1
            return True
        return False

## 📝 题目 1584：连接所有点的最小费用

::: {.callout-caution}
### 📖 题目描述
给你一个 `points` 数组，表示 2D 平面上的一些点，其中 `points[i] = [xi, yi]` 。

连接点 `[xi, yi]` 和点 `[xj, yj]` 的费用为它们之间的 **曼哈顿距离** ：`|xi - xj| + |yi - yj|` ，其中 `|val|` 表示 `val` 的绝对值。

请你返回将所有点连接起来（任意两点之间都可以建边）的最小总费用。要求连接后，任意两点之间有且仅有一条简单路径（即形成一棵树）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`points = [[0,0],[2,2],[3,10],[5,2],[7,0]]`
    * **输出**：`20`
    * **解释**：我们可以用总费用 20 将所有点连接起来。

* **示例 2**：
    * **输入**：`points = [[3,12],[-2,5],[-4,1]]`
    * **输出**：`18`
:::

In [9]:
class Solution1584:
    def minCostConnectPoints_Kruskal(self, points: List[List[int]]) -> int:
        n = len(points)
        uf = UnionFind(n)
        edges = []
        for i in range(n):  # 收集所有可能的边 (完全图)
            for j in range(i + 1, n):
                dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])  # 计算曼哈顿距离
                edges.append((dist, i, j))
        edges.sort()  # 贪心策略：按距离从小到大排序
        res, edges_added = 0, 0  # 遍历边，组装最小生成树
        for dist, i, j in edges:
            if uf.union(i, j):   # 只要 union 成功（没形成环），就采用这条边
                res += dist
                edges_added += 1
                if edges_added == n - 1:  # n 个点只需要 n - 1 条边就能连成树
                    break
        return res

    def minCostConnectPoints_Prim(self, points: List[List[int]]) -> int:
        n = len(points)
        if n <= 1:
            return 0
        adj = {i: [] for i in range(n)}  # 构建邻接表， adj[i] 存放点 i 能连出去的所有边：(距离, 邻居节点)
        for i in range(n):
            for j in range(i + 1, n):
                dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
                adj[i].append((dist, j))
                adj[j].append((dist, i))
        res = 0
        visited = set()
        min_heap = [(0, 0)]  # 最小堆：存放 (距离, 目标点)
        while len(visited) < n:
            dist, curr_node = heapq.heappop(min_heap)
            if curr_node in visited:  # 如果这个点已经在了，说明修这条路会形成环，直接废弃
                continue
            visited.add(curr_node)  # 标记为已使用，并加上费用
            res += dist
            for next_dist, next_node in adj[curr_node]:  # 把这个新节点能连出去的所有路，扔进堆里，作为未来的候选
                if next_node not in visited:  # 优化：只把还没使用的节点的边扔进去
                    heapq.heappush(min_heap, (next_dist, next_node))
        return res

In [27]:
#| code-fold: true
def test_1584(func1, func2):
    test_cases = [
        {"desc": "常规示例 1", "points": [[0,0],[2,2],[3,10],[5,2],[7,0]], "expected": 20},
        {"desc": "常规示例 2 (含负数)", "points": [[3,12],[-2,5],[-4,1]], "expected": 18},
        {"desc": "极限: 只有 1 个点", "points": [[0,0]], "expected": 0},
        {"desc": "极限: 只有 2 个点", "points": [[-5,-5],[5,5]], "expected": 20},
        {"desc": "四个点构成正方形", "points": [[0,0],[0,2],[2,0],[2,2]], "expected": 6},
        {"desc": "所有点在同一水平线上", "points": [[0,0],[1,0],[3,0],[6,0]], "expected": 6},
        {"desc": "所有点在同一垂直线上", "points": [[1,-2],[1,0],[1,5],[1,10]], "expected": 12},
        {"desc": "星型辐射 (中心点连所有点最优)", "points": [[0,0],[0,1],[0,-1],[1,0],[-1,0]], "expected": 4},
        {"desc": "存在坐标完全重叠的点", "points": [[1,1],[1,1],[2,2],[2,2]], "expected": 2}, # 0 + 2 + 0 = 2
        {"desc": "大跨度坐标", "points": [[-1000,-1000],[1000,1000]], "expected": 4000},
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'Kruskal':<4} | {'Prim':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual_1 = func1(tc["points"])
        actual_2 = func2(tc["points"])
        is_pass = actual_1 == actual_2 == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual_1:<4} | {actual_2:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1584(Solution1584().minCostConnectPoints_Kruskal, Solution1584().minCostConnectPoints_Prim)

ID   | 结果     | 预期   | Kruskal | Prim | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 20   | 20   | 20   | 常规示例 1
2    | ✅ PASS | 18   | 18   | 18   | 常规示例 2 (含负数)
3    | ✅ PASS | 0    | 0    | 0    | 极限: 只有 1 个点
4    | ✅ PASS | 20   | 20   | 20   | 极限: 只有 2 个点
5    | ✅ PASS | 6    | 6    | 6    | 四个点构成正方形
6    | ✅ PASS | 6    | 6    | 6    | 所有点在同一水平线上
7    | ✅ PASS | 12   | 12   | 12   | 所有点在同一垂直线上
8    | ✅ PASS | 4    | 4    | 4    | 星型辐射 (中心点连所有点最优)
9    | ✅ PASS | 2    | 2    | 2    | 存在坐标完全重叠的点
10   | ✅ PASS | 4000 | 4000 | 4000 | 大跨度坐标
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题没有直接给你“边”，而是给了你“点”。所以我们的第一步是**建图**：算出任意两个点之间的曼哈顿距离。如果有 $N$ 个点，就会有 $\frac{N(N-1)}{2}$ 条边。

#### ⚔️ 玩法一：Kruskal 算法（以“边”为核心，配合并查集）

1. **收集所有边**：把所有两点之间的边提取出来，存成 `(距离, 点A, 点B)` 的格式。
2. **按距离从小到大排序**：贪心思想，越便宜的边我越先考虑。
3. **并查集连线**：遍历排好序的边，如果 点A 和 点B **不在同一个集合**（即连上不会形成环），就选这条边！把它俩 `union` 起来，费用加上这条边的距离。
   *什么时候结束？* 当你成功连了 `N - 1` 条边时，所有点必定连通，直接提前 `break`！

#### 🛡️ 玩法二：Prim 算法（以“点”为核心，配合最小堆）

1. **随便挑一个起点**（比如点 0），把它标记为“已在树中”。
2. 把这个点连出去的**所有边** `(距离, 目标点)` 扔进一个**最小堆 (Min-Heap / `heapq`)** 里。
3. 只要堆不为空，就弹出堆顶（当前能摸到的最便宜的边）：
   - 如果这个边指向的“目标点”**已经在树中**了，说明是废边（会形成环），直接 `continue` 丢弃。
   - 如果“目标点”不在树中，太好了！费用加上这段距离，把这个目标点标记为“已在树中”。
   - 接着，把这个新加入的目标点连出去的所有边，也通通扔进最小堆里，继续扩张！
4. 当所有点都被标记为“已在树中”时，扩张结束。
:::

::: {.callout-note}
### 💡 复杂度分析
* **点数 $V$**：`len(points)`，**边数 $E$**：$O(V^2)$（完全图）。
* **Kruskal**：重点在排序，时间复杂度 $O(E \log E)$。适合边少的稀疏图。
* **Prim**：重点在堆操作，时间复杂度 $O(E \log V)$。在这道题（边极多的稠密图）中，Prim 理论上表现更好，但 Python 中 Kruskal 的 `list.sort()` 底层优化极佳，两者速度都很出色。
:::

## 📝 题目 1135：最低成本联通所有城市

::: {.callout-caution}
### 📖 题目描述
想象一下你是一名城市规划师，有 `n` 座城市，编号从 `1` 到 `n`。

给你一个 `connections` 数组，其中 `connections[i] = [city1, city2, cost]` 表示将 `city1` 和 `city2` 连接所要的成本（连接是双向的）。

请你返回连接所有城市的最少成本，使得任意两座城市之间都有一条路径相连。
**如果无法连接所有城市（即存在无法到达的孤岛），请返回 `-1`。**

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `connections = [[1,2,5],[1,3,6],[2,3,1]]`
    * **输出**：`6`
    * **解释**：
      选出 [1,2,5] 和 [2,3,1] 两条边，成本为 5 + 1 = 6。

* **示例 2**：
    * **输入**：`n = 4`, `connections = [[1,2,3],[3,4,4]]`
    * **输出**：`-1`
    * **解释**：
      城市 1, 2 连通，城市 3, 4 连通。但是没有任何线缆能把这两个阵营连起来，所以无法连通所有城市，返回 -1。
:::

In [44]:
class Solution1135:
    def minimumCost_Kruskal(self, n: int, connections: List[List[int]]) -> int:
        uf = UnionFind(n)
        edges = []
        for connection in connections:  # 收集所有边，转换为 (费用, 城市1, 城市2) 以便排序
            edges.append((connection[2], connection[0], connection[1]))
        edges.sort()  # 贪心：按费用从小到大排序
        res, edges_added = 0, 0
        for dist, i, j in edges:
            if uf.union(i - 1, j - 1):  # 偏移操作：题目城市是 1~n，并查集是 0~n-1
                res += dist
                edges_added += 1
                if edges_added == n - 1:  # n 个城市只需要 n-1 条边
                    break
        return res if edges_added == n - 1 else -1  # 如果最终凑够了 n-1 条边，说明全连通；否则说明有孤岛，返回 -1

    def minimumCost_Prim(self, n: int, connections: List[List[int]]) -> int:
        adj = {i: [] for i in range(1, n + 1)}  # 建图：初始化 1 到 n 的邻接表
        for connection in connections:
            adj[connection[0]].append((connection[2], connection[1]))
            adj[connection[1]].append((connection[2], connection[0]))
        res = 0
        visited = set()
        min_heap = [(0, 1)]  # 随便挑个城市（比如 1），到达它自己的费用是 0
        while min_heap and len(visited) < n:
            dist, curr_node = heapq.heappop(min_heap)
            if curr_node in visited:  # 遇到废边，直接跳过
                continue
            res += dist
            visited.add(curr_node)
            for next_dist, next_node in adj[curr_node]:  # 把新城市能连出去的、尚未占领的路，加入扩张候选堆
                heapq.heappush(min_heap, (next_dist, next_node))
        return res if len(visited) == n else -1

In [45]:
#| code-fold: true
def test_1135(func1, func2):
    test_cases = [
        {"desc": "示例 1: 常规连通", "n": 3, "connections": [[1,2,5],[1,3,6],[2,3,1]], "expected": 6},
        {"desc": "示例 2: 明显孤岛", "n": 4, "connections": [[1,2,3],[3,4,4]], "expected": -1},
        {"desc": "极限: 只有1个城市", "n": 1, "connections": [], "expected": 0},
        {"desc": "只有2个城市连通", "n": 2, "connections": [[1,2,100]], "expected": 100},
        {"desc": "只有2个城市但不连通", "n": 2, "connections": [], "expected": -1},
        {"desc": "星型连通 (中心发散)", "n": 4, "connections": [[1,2,1],[1,3,2],[1,4,3]], "expected": 6},
        {"desc": "包含平行边 (同样的两点有不同报价)", "n": 3, "connections": [[1,2,10],[1,2,1],[1,3,5]], "expected": 6}, # 取 1 和 5
        {"desc": "隐藏的孤岛 (连通分量内部成环)", "n": 4, "connections": [[1,2,1],[2,3,1],[3,1,1]], "expected": -1}, # 4 号是孤儿
        {"desc": "直线长距离连通", "n": 5, "connections": [[1,2,1],[2,3,1],[3,4,1],[4,5,1]], "expected": 4},
        {"desc": "多余的极长边", "n": 3, "connections": [[1,2,1],[2,3,1],[1,3,1000]], "expected": 2},
        {"desc": "两个独立的三角形", "n": 6, "connections": [[1,2,1],[2,3,1],[3,1,1],[4,5,1],[5,6,1],[6,4,1]], "expected": -1}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'Kruskal':<4} | {'Prim':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual_1 = func1(tc["n"], tc["connections"])
        actual_2 = func2(tc["n"], tc["connections"])
        is_pass = actual_1 == actual_2 == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual_1:<4} | {actual_2:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1135(Solution1135().minimumCost_Kruskal, Solution1135().minimumCost_Prim)

ID   | 结果     | 预期   | Kruskal | Prim | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 6    | 6    | 6    | 示例 1: 常规连通
2    | ✅ PASS | -1   | -1   | -1   | 示例 2: 明显孤岛
3    | ✅ PASS | 0    | 0    | 0    | 极限: 只有1个城市
4    | ✅ PASS | 100  | 100  | 100  | 只有2个城市连通
5    | ✅ PASS | -1   | -1   | -1   | 只有2个城市但不连通
6    | ✅ PASS | 6    | 6    | 6    | 星型连通 (中心发散)
7    | ✅ PASS | 6    | 6    | 6    | 包含平行边 (同样的两点有不同报价)
8    | ✅ PASS | -1   | -1   | -1   | 隐藏的孤岛 (连通分量内部成环)
9    | ✅ PASS | 4    | 4    | 4    | 直线长距离连通
10   | ✅ PASS | 2    | 2    | 2    | 多余的极长边
11   | ✅ PASS | -1   | -1   | -1   | 两个独立的三角形
-----------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解

这道题是极其纯粹的最小生成树（MST）模板题，没有任何花哨的建图魔法。核心考验点只有一个：**如何精准判断出图里存在“永远连不上”的孤岛城市？**

#### ⚔️ 玩法一：Kruskal 算法 (贪心排序 + 并查集)
Kruskal 像是一个精打细算的包工头，把所有修路方案按价格从低到高排好，然后挨个尝试：

1. **收集与排序**：把 `connections` 里的边提取成 `(cost, u, v)` 并 `sort()`。
2. **巧妙避开索引坑**：题目给的城市是 `1` 到 `n`，而常规初始化的 `UnionFind(n)` 是 `0` 到 `n-1`。在合并时使用 `uf.union(u - 1, v - 1)` 完美对齐索引！
3. **孤岛判别法（核心）**：`n` 个城市完全连通需要且仅需要 `n - 1` 条边。我们维护一个 `edges_added` 计数器。如果遍历完所有的边，计数器依然 `< n - 1`，说明即使把能修的路全修了也连不通，直接返回 `-1`。

#### 🛡️ 玩法二：Prim 算法 (优先队列 + 帝国扩张)
Prim 像是一个从首都（比如城市 `1`）向外不断吞并领土的帝国：

1. **精准建图**：由于城市从 `1` 开始，构建邻接表必须严格使用 `adj = {i: [] for i in range(1, n + 1)}`。
2. **带资进组**：将起点 `(0, 1)`（花费 0 元到达城市 1）丢进最小堆 `min_heap`。
3. **孤岛判别法（核心）**：只要堆不为空且版图还没统一（`len(visited) < n`），就持续弹出最便宜的路并占领新城市。
   *如果堆都空了（能摸到的路全修完了），循环被迫终止，但此时 `len(visited)` 依然 `< n`，说明剩下的城市是大海中的孤岛，直接返回 `-1`！*
:::

::: {.callout-note}
### 💡 复杂度分析
假设有 $V$ 个城市（点），$E$ 条连接方案（边）。

* **Kruskal**：时间 $O(E \log E)$（瓶颈在排序），空间 $O(V + E)$。
* **Prim**：时间 $O(E \log V)$（堆操作），空间 $O(V + E)$。
:::

## 📝 题目 1168：水资源分配优化

::: {.callout-caution}
### 📖 题目描述
村里面一共有 `n` 栋房子，我们希望通过建造水井和铺设管道来为所有房子供水。

对于每个房子 `i`，我们有两种方式为它供水：
1. **自己建水井**：直接在第 `i` 栋房子建水井的成本记录在数组 `wells` 中，其中 `wells[i-1]` 是建井的费用。
2. **铺设管道**：从另一个有水的房子接水管过来。数组 `pipes` 记录了管道信息，其中 `pipes[j] = [house1, house2, cost]` 代表用成本 `cost` 连接 `house1` 和 `house2`。

请你计算并返回为**所有房子**都供水的 **最低总成本**。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `wells = [1, 2, 2]`, `pipes = [[1, 2, 1], [2, 3, 1]]`
    * **输出**：`3`
    * **解释**：
      房子 1 自己建水井，花费 1（此时房子 1 有水了）。
      铺设管道连接房子 1 和 2，花费 1（此时房子 2 有水了）。
      铺设管道连接房子 2 和 3，花费 1（此时房子 3 有水了）。
      总花费：1 + 1 + 1 = 3。

* **示例 2**：
    * **输入**：`n = 2`, `wells = [1, 1]`, `pipes = [[1, 2, 1]]`
    * **输出**：`2`
    * **解释**：
      方案一：房子 1 和 2 各自建水井，花费 1 + 1 = 2。
      方案二：房子 1 建水井花费 1，连接房子 1 和 2 花费 1，总计 1 + 1 = 2。
      最低总成本为 2。

* **示例 3**：
    * **输入**：`n = 2`, `wells = [1, 5]`, `pipes = [[1, 2, 10]]`
    * **输出**：`6`
    * **解释**：
      铺设管道的成本 (10) 远大于房子 2 自己建井的成本 (5)。
      因此，房子 1 和房子 2 各自建水井是最优解，花费 1 + 5 = 6。
:::

In [20]:
class Solution1168:
    def minCostToSupplyWater_Kruskal(self, n: int, wells: List[int], pipes: List[List[int]]) -> int:
        uf = UnionFind(n + 1)
        edges = []
        for i in range(1, n + 1):  # 虚拟节点 0 到各个房子的打井边
            edges.append((wells[i - 1], 0, i))
        for pipe in pipes:  # 房子之间的管道边，统一元组格式为 (cost, u, v)
            edges.append((pipe[2], pipe[0], pipe[1]))
        edges.sort()  # 贪心排序
        res, edges_added = 0, 0
        for dist, i, j in edges:
            if uf.union(i, j):
                res += dist
                edges_added += 1
                if edges_added == n:  # N+1 个点需要 N 条边
                    break
        return res

    def minCostToSupplyWater_Prim(self, n: int, wells: List[int], pipes: List[List[int]]) -> int:
        adj = {i: [] for i in range(n + 1)}
        for i in range(1, n + 1):  # 虚拟节点 0 到各个房子的打井边
            adj[0].append((wells[i - 1], i))
            adj[i].append((wells[i - 1], 0))
        for pipe in pipes:  # 房子之间的管道双向边
            adj[pipe[0]].append((pipe[2], pipe[1]))
            adj[pipe[1]].append((pipe[2], pipe[0]))
        res = 0
        visited = set()
        min_heap = [(0, 0)]  # (距离, 节点)
        while len(visited) < n + 1:
            dist, curr_node = heapq.heappop(min_heap)
            if curr_node in visited:
                continue
            visited.add(curr_node)
            res += dist
            for next_dist, next_node in adj[curr_node]:
                if next_node not in visited:  # 性能优化：不在 visited 里的才扔进堆
                    heapq.heappush(min_heap, (next_dist, next_node))
        return res

In [28]:
#| code-fold: true
def test_1168(func1, func2):
    test_cases = [
        {"desc": "示例 1: 1打井, 连2连3", "n": 3, "wells": [1,2,2], "pipes": [[1,2,1],[2,3,1]], "expected": 3},
        {"desc": "示例 2: 方案等价", "n": 2, "wells": [1,1], "pipes": [[1,2,1]], "expected": 2},
        {"desc": "示例 3: 管道太贵各自打井", "n": 2, "wells": [1,5], "pipes": [[1,2,10]], "expected": 6},
        {"desc": "极限: 只有1个房子", "n": 1, "wells": [10], "pipes": [], "expected": 10},
        {"desc": "管道比打井贵很多", "n": 4, "wells": [1,1,1,1], "pipes": [[1,2,100],[2,3,100],[3,4,100]], "expected": 4},
        {"desc": "一口神井养活全村", "n": 4, "wells": [1,100,100,100], "pipes": [[1,2,1],[1,3,1],[1,4,1]], "expected": 4},
        {"desc": "含多余废弃管道 (闭环)", "n": 3, "wells": [2,2,2], "pipes": [[1,2,1],[2,3,1],[3,1,10]], "expected": 4}, # 1井(2)+管(1)+管(1)
        {"desc": "不连通的子图只能打多口井", "n": 4, "wells": [5,5,5,5], "pipes": [[1,2,1],[3,4,1]], "expected": 12}, # 1井(5)+1-2(1), 3井(5)+3-4(1)
        {"desc": "平行管道取最小", "n": 2, "wells": [10,10], "pipes": [[1,2,5],[1,2,1],[1,2,8]], "expected": 11}, # 1井(10) + 最便宜管道(1)
        {"desc": "管道免费", "n": 3, "wells": [100,200,300], "pipes": [[1,2,0],[2,3,0]], "expected": 100}, # 1井(100) + 管(0) + 管(0)
        {"desc": "线性长村落接力传递", "n": 5, "wells": [1,10,10,10,10], "pipes": [[1,2,1],[2,3,2],[3,4,1],[4,5,2]], "expected": 7} # 井(1)+管和(6)
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'Kruskal':<4} | {'Prim':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual_1 = func1(tc["n"], tc["wells"], tc["pipes"])
        actual_2 = func2(tc["n"], tc["wells"], tc["pipes"])
        is_pass = actual_1 == actual_2 == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual_1:<4} | {actual_2:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1168(Solution1168().minCostToSupplyWater_Kruskal, Solution1168().minCostToSupplyWater_Prim)

ID   | 结果     | 预期   | Kruskal | Prim | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 3    | 3    | 3    | 示例 1: 1打井, 连2连3
2    | ✅ PASS | 2    | 2    | 2    | 示例 2: 方案等价
3    | ✅ PASS | 6    | 6    | 6    | 示例 3: 管道太贵各自打井
4    | ✅ PASS | 10   | 10   | 10   | 极限: 只有1个房子
5    | ✅ PASS | 4    | 4    | 4    | 管道比打井贵很多
6    | ✅ PASS | 4    | 4    | 4    | 一口神井养活全村
7    | ✅ PASS | 4    | 4    | 4    | 含多余废弃管道 (闭环)
8    | ✅ PASS | 12   | 12   | 12   | 不连通的子图只能打多口井
9    | ✅ PASS | 11   | 11   | 11   | 平行管道取最小
10   | ✅ PASS | 100  | 100  | 100  | 管道免费
11   | ✅ PASS | 7    | 7    | 7    | 线性长村落接力传递
-----------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解

这道题最大的难点在于“点”（打井）和“边”（铺管道）都带有费用，无法直接套用 MST 模板。
**图论建模神技：引入“虚拟节点 0”（超级源点）！**
想象存在一个拥有无限水量的“0号水库”。在房子 `i` 花费 `wells[i-1]` 打井，等价于**从“0号水库”修一条花费为 `wells[i-1]` 的管道到房子 `i`**。
这样一来，所有的“打井操作”全部变成了“边”！原问题完美转化为：**在一个包含 `0` 到 `n`（共 `n+1` 个点）的图中，寻找最小生成树。**

#### ⚔️ 玩法一：Kruskal 算法（以“边”为核心）
1. **收集边**：
   - 遍历 `wells`，把每个房子的打井费用变成边 `(wells[i-1], 0, i)` 加入边集。
   - 遍历 `pipes`，把每条管道变成边 `(cost, house1, house2)` 加入边集。
2. **贪心排序**：将所有边按费用从小到大 `sort()`。
3. **并查集连线**：遍历排好序的边，用 `union(u, v)` 判断是否成环。不成环就采用这条边，累加费用。
   - *注意*：现在图里有 `n+1` 个点，所以当连够了 **`n` 条边** 时，整棵树就建成了！

#### 🛡️ 玩法二：Prim 算法（以“点”为核心）
1. **建图 (邻接表)**：
   - 创建包含 `0` 到 `n` 的邻接表 `adj`。
   - 把打井费用作为 `0` 和 `i` 之间的双向边加入 `adj`。
   - 把管道作为 `house1` 和 `house2` 之间的双向边加入 `adj`。
2. **帝国扩张**：
   - 从虚拟节点 `0` 开始扩张（`heappush(min_heap, (0, 0))`），费用为 0。
   - 只要弹出的点不在 `visited` 集合中，就将其占领，累加费用，并把它连出去的所有新边扔进最小堆。
   - 当 `len(visited) == n + 1` 时，所有节点（包含水库和所有房子）全部通水，扩张结束！
:::

::: {.callout-note}
### 💡 复杂度分析
假设房子数量为 $N$，管道数量为 $M$。转化后，总节点数 $V = N + 1$，总边数 $E = N + M$。

* **Kruskal 算法**：
  - **时间复杂度**：$O(E \log E)$。瓶颈在于对所有边进行排序。
  - **空间复杂度**：$O(V + E)$。需要存储边集以及并查集的 `parent` 数组。

* **Prim 算法**：
  - **时间复杂度**：$O(E \log V)$。每条边最多入堆出堆一次。
  - **空间复杂度**：$O(V + E)$。需要存储邻接表以及最小堆。
:::